# 02 Inventory

Scan a sandbox folder recursively and write CSV / Parquet inventory outputs. No file changes are made.

In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

POLICY_PATH = PROJECT_ROOT / 'policy' / 'SCH_fileserver_policy_v2_5.yaml'
print('PROJECT_ROOT =', PROJECT_ROOT)
print('POLICY_PATH =', POLICY_PATH)


PROJECT_ROOT = c:\00_Developement\sch-file-organizer
POLICY_PATH = c:\00_Developement\sch-file-organizer\policy\SCH_fileserver_policy_v2_5.yaml


In [2]:
from src.inventory import build_inventory, save_inventory, summarize_inventory
from src.policy_loader import PolicyLoader

policy = PolicyLoader.from_file(POLICY_PATH)
OUTPUT_DIR = PROJECT_ROOT / 'data' / 'outputs'
WORKING_DIR = PROJECT_ROOT / 'data' / 'working'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
WORKING_DIR.mkdir(parents=True, exist_ok=True)


In [3]:
# # Optional: create a tiny demo sandbox for smoke testing
# DEMO_ROOT = WORKING_DIR / 'demo_sandbox'
# DEMO_ROOT.mkdir(parents=True, exist_ok=True)
# (DEMO_ROOT / 'notes').mkdir(exist_ok=True)
# (DEMO_ROOT / 'trash').mkdir(exist_ok=True)
# (DEMO_ROOT / 'notes' / 'readme.txt').write_text('demo file\n', encoding='utf-8')
# (DEMO_ROOT / 'trash' / 'Thumbs.db').write_text('junk\n', encoding='utf-8')
# (DEMO_ROOT / 'notes' / 'dup1.txt').write_text('same\n', encoding='utf-8')
# (DEMO_ROOT / 'notes' / 'dup2.txt').write_text('same\n', encoding='utf-8')
# print('Demo sandbox ready at', DEMO_ROOT)


In [4]:
# SCAN_ROOT = DEMO_ROOT  # replace with your copied sandbox root after smoke test
SCAN_ROOT = Path(r'C:\Users\User\Desktop\Random_Files_WORKING_COPY')  # <-- set this to your actual scan root
print('Scanning', SCAN_ROOT)
inv = build_inventory(SCAN_ROOT)
summary = summarize_inventory(inv)
summary


Scanning C:\Users\User\Desktop\Random_Files_WORKING_COPY


{'file_count': 1806,
 'total_size_bytes': 589569619,
 'duplicate_files': 260,
 'duplicate_groups': 130,
 'max_path_length': 297,
 'max_depth_segments': 4}

In [5]:
from datetime import datetime

stamp = datetime.now().strftime('%Y%m%d_%H%M%S')
output_base = OUTPUT_DIR / f'inventory_{SCAN_ROOT.name}_{stamp}'
csv_path, parquet_path = save_inventory(inv, output_base)
print('CSV:', csv_path)
print('Parquet:', parquet_path)


CSV: c:\00_Developement\sch-file-organizer\data\outputs\inventory_Random_Files_WORKING_COPY_20260308_115612.csv
Parquet: c:\00_Developement\sch-file-organizer\data\outputs\inventory_Random_Files_WORKING_COPY_20260308_115612.parquet


In [6]:
inv.head(5)

,scan_root,absolute_path,relative_path,parent_relative,filename,stem,suffix,size_bytes,modified_at,created_at,depth_segments,path_length,filename_length,is_hidden,is_symlink,top_segment,hash,is_duplicate_hash,duplicate_group_size
0,C:\Users\User\Desktop\Random_Files_WORKING_COPY,C:\Users\User\Desktop\Random_Files_WORKING_COP...,01_selected\New Text Document.ogb,01_selected,New Text Document.ogb,New Text Document,.ogb,0,2026-03-08 09:44:01.994213820,2026-03-08 09:44:01.994213820,2,81,21,False,False,01_selected,cae66941d9efbd404e4d88758ea67670,False,1
1,C:\Users\User\Desktop\Random_Files_WORKING_COPY,C:\Users\User\Desktop\Random_Files_WORKING_COP...,01_selected\doclaynet_pdf\doclaynet_pdf_0001_r...,01_selected\doclaynet_pdf,doclaynet_pdf_0001_refman-8.0-en_p3115.pdf,doclaynet_pdf_0001_refman-8.0-en_p3115,.pdf,16575,2026-03-08 04:32:19.136427641,2026-03-08 09:04:28.638393402,3,116,42,False,False,01_selected,be97ade4b1299dd340b73aacfdbabebb,False,1
2,C:\Users\User\Desktop\Random_Files_WORKING_COPY,C:\Users\User\Desktop\Random_Files_WORKING_COP...,01_selected\doclaynet_pdf\doclaynet_pdf_0001_r...,01_selected\doclaynet_pdf,doclaynet_pdf_0001_refman-8.0-en_p3115.txt,doclaynet_pdf_0001_refman-8.0-en_p3115,.txt,3165,2026-03-08 04:32:19.136427641,2026-03-08 09:04:28.638393402,3,116,42,False,False,01_selected,2c3f4221d08c2dd7c7050480124cf1d6,False,1
3,C:\Users\User\Desktop\Random_Files_WORKING_COPY,C:\Users\User\Desktop\Random_Files_WORKING_COP...,01_selected\doclaynet_pdf\doclaynet_pdf_0002_p...,01_selected\doclaynet_pdf,doclaynet_pdf_0002_pilot_handbook_p257.pdf,doclaynet_pdf_0002_pilot_handbook_p257,.pdf,27076,2026-03-08 04:32:19.136427641,2026-03-08 09:04:28.653279305,3,116,42,False,False,01_selected,fa5b1b0778c3752db1708ef8c5aafd93,False,1
4,C:\Users\User\Desktop\Random_Files_WORKING_COPY,C:\Users\User\Desktop\Random_Files_WORKING_COP...,01_selected\doclaynet_pdf\doclaynet_pdf_0002_p...,01_selected\doclaynet_pdf,doclaynet_pdf_0002_pilot_handbook_p257.txt,doclaynet_pdf_0002_pilot_handbook_p257,.txt,5,2026-03-08 04:32:19.136427641,2026-03-08 09:04:28.660314083,3,116,42,False,False,01_selected,74a0cb4236beded2e41019909a407677,False,1


In [7]:
inv.groupby('suffix', dropna=False).size().sort_values(ascending=False).to_frame('count').head(20)

,count
suffix,
.pdf,682
.txt,353
.xls,213
.doc,165
.stp,156
.html,86
.png,75
.ppt,50
.csv,17


In [8]:
inv[inv['is_duplicate_hash']].sort_values(['hash', 'relative_path'])[['relative_path', 'hash', 'duplicate_group_size']]

,relative_path,hash,duplicate_group_size
51,01_selected\doclaynet_pdf\doclaynet_pdf_0026_1...,024932feb957cee6760d1024d043d1bc,2
1355,01_selected\random_1_dedup_target\doclaynet_pd...,024932feb957cee6760d1024d043d1bc,2
380,01_selected\funsd_scanned\funsd_scanned_0020_4...,03639c48a0e2533172cfcbbc81087976,2
1404,01_selected\random_1_dedup_target\funsd_scanne...,03639c48a0e2533172cfcbbc81087976,2
48,01_selected\doclaynet_pdf\doclaynet_pdf_0024_a...,037d6df5666e865cd74ae8b1e9fedc47,2
...,...,...,...
1328,01_selected\random_1_dedup_target\doclaynet_pd...,fc9237a6c729a93816a129dc73dde132,2
468,01_selected\fusion360\fusion360_0008_31663_c1a...,fea5de6909fada9370e7f06af9aaa643,2
1438,01_selected\random_1_dedup_target\fusion360_00...,fea5de6909fada9370e7f06af9aaa643,2
379,01_selected\funsd_scanned\funsd_scanned_0020_4...,fedf27f64f91931960ead67750cfdd1a,2


In [9]:
inv.sort_values('path_length', ascending=False)[['relative_path', 'path_length', 'filename_length']].head(20)

,relative_path,path_length,filename_length
1805,01_selected\very_very_very_very_very_very_very...,297,35
1785,01_selected\very_very_very_very_very_very_very...,297,35
1804,01_selected\very_very_very_very_very_very_very...,297,35
1769,01_selected\very_very_very_very_very_very_very...,297,35
1770,01_selected\very_very_very_very_very_very_very...,297,35
1772,01_selected\very_very_very_very_very_very_very...,297,35
1771,01_selected\very_very_very_very_very_very_very...,297,35
1767,01_selected\very_very_very_very_very_very_very...,297,35
1768,01_selected\very_very_very_very_very_very_very...,297,35
1753,01_selected\very_very_very_very_very_very_very...,297,35
